In [8]:
# Install dependencies
# %pip install kagglehub pandas

In [31]:
import kagglehub
import pandas as pd
import os

In [32]:
path = kagglehub.dataset_download("shivamb/netflix-shows")

In [33]:
df = pd.read_csv(os.path.join(path, "netflix_titles.csv"))

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [35]:
df.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [36]:
(df.isna().mean() * 100).round(1)

show_id          0.0
type             0.0
title            0.0
director        29.9
cast             9.4
country          9.4
date_added       0.1
release_year     0.0
rating           0.0
duration         0.0
listed_in        0.0
description      0.0
dtype: float64

In [37]:
df.duplicated().sum()

np.int64(0)

In [38]:
df.describe(include="all")

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8807.000000,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,NaN,17,220,514,8775
top,s1,Movie,Dick Johnson Is Dead,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",NaN,TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,NaN,3207,1793,362,4
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.180198,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.819312,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN


In [39]:
df["type"].value_counts()

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

In [40]:
df_clean = df.copy()

In [41]:
df_clean["director"] = df_clean["director"].fillna("Unknown")

In [42]:
extraido = df_clean["duration"].str.extract(r"(\d+)\s*(\w+)")

In [43]:
df_clean["duration_num"] = pd.to_numeric(extraido[0])

In [44]:
df_clean["duration_min"] = df_clean["duration_num"].where(df_clean["type"] == "Movie")

In [45]:
df_clean["seasons"] = df_clean["duration_num"].where(df_clean["type"] == "TV Show")

In [46]:
df_clean["date_added"] = pd.to_datetime(df_clean["date_added"].str.strip(), format="%B %d, %Y")

In [47]:
df_clean["year_added"] = df_clean["date_added"].dt.year

In [48]:
df_clean["month_added"] = df_clean["date_added"].dt.month

In [49]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      8807 non-null   str           
 4   cast          7982 non-null   str           
 5   country       7976 non-null   str           
 6   date_added    8797 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8803 non-null   str           
 9   duration      8804 non-null   str           
 10  listed_in     8807 non-null   str           
 11  description   8807 non-null   str           
 12  duration_num  8804 non-null   float64       
 13  duration_min  6128 non-null   float64       
 14  seasons       2676 non-null   float64       
 15  year_added    8797 non-null   float64       
 16 

In [50]:
df_clean.isna().sum()

show_id            0
type               0
title              0
director           0
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
duration_num       3
duration_min    2679
seasons         6131
year_added        10
month_added       10
dtype: int64

In [51]:
df_clean[df_clean["duration"].isna()][["title", "type", "rating", "duration"]]

,title,type,rating,duration
5541,Louis C.K. 2017,Movie,74 min,NaN
5794,Louis C.K.: Hilarious,Movie,84 min,NaN
5813,Louis C.K.: Live at the Comedy Store,Movie,66 min,NaN


In [52]:
mask = (df_clean["duration"].isna()) & (df_clean["rating"].str.contains("min", na=False))
df_clean[mask]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_num,duration_min,seasons,year_added,month_added
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,2017-04-04,2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi...",NaN,NaN,NaN,2017.0,4.0
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,2016-09-16,2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...,NaN,NaN,NaN,2016.0,9.0
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,2016-08-15,2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...,NaN,NaN,NaN,2016.0,8.0


In [53]:
df_clean.loc[mask, "duration"] = df_clean.loc[mask, "rating"]
df_clean.loc[mask, "rating"] = "Unknown"
df_clean.loc[[5541, 5794, 5813], ["title", "rating", "duration", "duration_num", "duration_min"]]

,title,rating,duration,duration_num,duration_min
5541,Louis C.K. 2017,Unknown,74 min,NaN,NaN
5794,Louis C.K.: Hilarious,Unknown,84 min,NaN,NaN
5813,Louis C.K.: Live at the Comedy Store,Unknown,66 min,NaN,NaN


In [54]:
extraido = df_clean["duration"].str.extract(r"(\d+)\s*(\w+)")
df_clean["duration_num"] = pd.to_numeric(extraido[0])

df_clean["duration_min"] = df_clean["duration_num"].where(df_clean["type"] == "Movie")
df_clean["seasons"] = df_clean["duration_num"].where(df_clean["type"] == "TV Show")

In [56]:
df_clean.loc[[5541, 5794, 5813], ["title", "duration", "duration_num", "duration_min"]]

,title,duration,duration_num,duration_min
5541,Louis C.K. 2017,74 min,74,74.0
5794,Louis C.K.: Hilarious,84 min,84,84.0
5813,Louis C.K.: Live at the Comedy Store,66 min,66,66.0


In [60]:
df_clean["rating"] = df_clean["rating"].fillna("Unknown")

In [61]:
df_clean.loc[[5541, 5794, 5813], ["title", "duration", "duration_num", "duration_min"]]

,title,duration,duration_num,duration_min
5541,Louis C.K. 2017,74 min,74,74.0
5794,Louis C.K.: Hilarious,84 min,84,84.0
5813,Louis C.K.: Live at the Comedy Store,66 min,66,66.0


In [64]:
df_clean.isna().sum()

show_id            0
type               0
title              0
director           0
cast             825
country          831
date_added        10
release_year       0
rating             0
duration           0
listed_in          0
description        0
duration_num       0
duration_min    2676
seasons         6131
year_added        10
month_added       10
dtype: int64

In [65]:
df_clean["cast"] = df_clean["cast"].fillna("Unknown")
df_clean["country"] = df_clean["country"].fillna("Unknown")

In [68]:
df_clean[df_clean["date_added"].isna()]["type"].value_counts()

type
TV Show    10
Name: count, dtype: int64

In [70]:
df_clean.duplicated().sum()

np.int64(0)

In [72]:
df_clean.dtypes

show_id                    str
type                       str
title                      str
director                   str
cast                       str
country                    str
date_added      datetime64[us]
release_year             int64
rating                     str
duration                   str
listed_in                  str
description                str
duration_num             int64
duration_min           float64
seasons                float64
year_added             float64
month_added            float64
dtype: object

In [78]:
df_clean["year_added"] = df_clean["year_added"].astype("Int64")
df_clean["month_added"] = df_clean["month_added"].astype("Int64")
df_clean["seasons"] = df_clean["seasons"].astype("Int64")
df_clean["duration_min"] = df_clean["duration_min"].astype("Int64")

In [77]:
df_clean.duplicated().sum()

np.int64(0)

In [80]:
df_clean["rating"].value_counts()

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
Unknown        7
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

In [81]:
df_clean["rating"] = df_clean["rating"].replace({"UR": "NR"})